In [ ]:
import matplotlib.pyplot as plt
import scienceplots
from tqdm import tqdm

from e2e.e2esim_uncert import run_e2e_prediction_with_uncertainty
from pathlib import Path

from e2e.e2esim_uncert import plot_outline_with_uncertainty_envelope

import numpy as np
import torch

from e2e.data.resampled import ResampledE2EDataset
from e2e.helpers import timing
from e2e.helpers.wandb import download_model
from e2e.models.mlp import ResMLP
from e2e.models.model import Model
from e2e.models.modelV2 import ModelV2

timing.ENABLE_TIMING = True
from e2e.data.loader import EXPERIMENT as EXP
from e2e.models.recurrent import LSTM
from e2e.prediction.end_to_end import DataLoader, Plotter


plt.style.use(['science', 'ieee', 'grid'])
plt.rcParams.update({"font.size": 12})

%load_ext autoreload
%autoreload 2

In [ ]:
MAC_DATA_DIR = Path("/Users/magnus/datasets/WAAM/test")
MLP_INPUT_LENGTH = 224
data_loader = DataLoader(MAC_DATA_DIR)
cross_section_samples = data_loader.load_samples([EXP.RANDOM_EX6])
dataset = ResampledE2EDataset(mirror=False, segment_length=MLP_INPUT_LENGTH).create(cross_section_samples)

In [ ]:
# GLOBAL
VERSION = "best"
DEVICE = "cpu"

# MLP
MLP_PROJECT_NAME = "waam-e2e-footprint"
MLP_RUN_ID = "y21szkna"

MLP_INPUT_LENGTH = 224
MLP_HIDDEN_LENGTH_FACTOR = 2
MLP_TARGET_LENGTH = 2
MLP_N_LAYERS = 3
MLP_P = 0.5

# LSTM
LSTM_INPUT_LENGTH = 90
LSTM_TARGET_LENGTH = 90
LSTM_P = 0.9

N_PREDICTIONS = 30

mlp = ResMLP(
    p=MLP_P,
    n_input_features=MLP_INPUT_LENGTH,
    n_hidden=MLP_INPUT_LENGTH * MLP_HIDDEN_LENGTH_FACTOR,
    n_output_features=MLP_TARGET_LENGTH,
    n_layers=MLP_N_LAYERS,
)

lstm = LSTM(
    p=LSTM_P,
    n_input_features=LSTM_INPUT_LENGTH,
    n_output_features=LSTM_TARGET_LENGTH,
    n_hidden=LSTM_TARGET_LENGTH * 3,
    n_layers=10,
)

# instantiate models

footprint_path = download_model(project=MLP_PROJECT_NAME, run_id=MLP_RUN_ID, version=VERSION)

footprint_predictor = Model.load_from_checkpoint(
    model=mlp, checkpoint_path=footprint_path, map_location=torch.device(DEVICE)
)

base = Path("/Users/magnus/repos/waam-e2e/e2e/")
lstm_path = base / Path("./waam-e2e-pre/mbw4tdhh/checkpoints/epoch=199-step=2400.ckpt")
shape_predictor = ModelV2.load_from_checkpoint(
    model=lstm, checkpoint_path=lstm_path, map_location=torch.device(DEVICE)
)
shape_predictor.to("cpu")

In [ ]:
xts = []
zs = []

start_idx = 279
hatch_distance = 52
offset = hatch_distance // 2

L = 10
for l in range(L+1):
    zs.extend(np.array([l]*(L-l)))
    layer = np.arange(start_idx, start_idx + hatch_distance * (L-l), hatch_distance, dtype=int)
    xts.extend(layer)
    start_idx += offset

xts = np.array(xts)
zs = np.array(zs)

In [ ]:
plt.scatter(xts, zs, color="red", marker="x", alpha=1)

In [6]:
SIMUlATION_RUNS = 300

dataset.torchpositions = xts

outlines = []

for iter in tqdm(range(SIMUlATION_RUNS)):
    try:
        dataset = run_e2e_prediction_with_uncertainty(footprint_predictor, shape_predictor, dataset, N_PREDICTIONS=1)
        outline = np.array(dataset.predictions).max(axis=0)[200:901]
    except:
        print("failed")
        continue
    dataset.predictions = []
    outlines.append(outline)


100%|██████████| 300/300 [00:38<00:00,  7.79it/s]


In [7]:
outlines = np.array(outlines)
outlines.shape

(300, 701)

In [8]:
mean_outline = outlines.mean(axis=0)
std_outline = outlines.std(axis=0)

In [13]:
!pwd

/Users/magnus/repos/waam-e2e/e2e


In [15]:
outputs = Path("./outputs")
sim = outputs / "sim"
# save mean outline
np.save(sim / "mean_outline.npy", mean_outline)
np.save(sim / "std_outline.npy", std_outline)

In [9]:
start = 40
end = 670

In [10]:
ts_zero = []
for gt in dataset.ground_truth:
    t = gt[200:901]
    offset = np.mean(t[:40])
    t = t - offset
    ts_zero.append(t)
    
ts = np.array(ts_zero)
outline_true = ts.max(axis=0)

In [ ]:
fig = plot_outline_with_uncertainty_envelope(mean_outline, std_outline)

2023-10-26 09:56:36,010 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-26 09:56:36,039 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-26 09:56:36,116 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-26 09:56:36,168 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-26 09:56:36,266 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-26 09:56:36,268 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-26 09:56:36,278 - MainThread - INFO - matplotlib.texmanager - No LaT

In [66]:
e2esim = Path("/Users/magnus/Desktop/graphics/results/e2esim")
fig.savefig(e2esim / "e2esim_uncertainty_envelope.pdf", bbox_inches="tight")

2023-10-25 15:58:09,852 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-25 15:58:09,856 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-25 15:58:09,862 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-25 15:58:09,867 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-25 15:58:09,871 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-25 15:58:09,875 - MainThread - INFO - matplotlib.texmanager - No LaTeX-compatible font found for the serif fontfamily in rcParams. Using default.
2023-10-25 15:58:09,880 - MainThread - INFO - matplotlib.texmanager - No LaT